In [12]:
import os, sys
if "google.colab" in sys.modules and not os.path.isdir("../data/raw"):
    if not os.path.isdir("/content/Busqueda-Empleo"):
        !git clone https://github.com/samuelvelez1003/Busqueda-Empleo.git /content/Busqueda-Empleo
    %cd /content/Busqueda-Empleo/notebooks
print("Carpeta de trabajo:", os.getcwd())

Cloning into '/content/Busqueda-Empleo'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 129 (delta 9), reused 7 (delta 3), pack-reused 112 (from 2)
Receiving objects: 100% (129/129), 54.85 MiB | 9.89 MiB/s, done.
Resolving deltas: 100% (27/27), done.
Updating files: 100% (62/62), done.
/content/Busqueda-Empleo/notebooks
Carpeta de trabajo: /content/Busqueda-Empleo/notebooks


## 1. Datos de entrada
En `data/raw/geih_2025/` hay una carpeta por mes con **solo los 4 módulos que necesitamos** (los CSV originales del DANE, sin modificar):

| Archivo | Módulo del DANE | Qué usamos |
|---|---|---|
| `caracteristicas_generales.csv` | Características generales, seguridad social en salud y educación | Edad, sexo, nivel educativo, si estudia, ciudad, factor de expansión |
| `fuerza_de_trabajo.csv` | Fuerza de trabajo | Quién está en la fuerza de trabajo |
| `ocupados.csv` | Ocupados | Meses en el empleo actual, contrato, pensión, ingreso, sector |
| `no_ocupados.csv` | No ocupados | Si está desocupado y si **ha trabajado antes** |

Los módulos se unen con la llave de persona: **DIRECTORIO + SECUENCIA_P + ORDEN** (más el mes).

In [13]:
import os, glob
import pandas as pd
import numpy as np

# Mostrar tablas completas (Colab y Jupyter recortan filas, columnas y textos largos)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# Carpeta de datos del repositorio
for d in ["../data", "data", "../Data", "Data"]:
    if os.path.isdir(os.path.join(d, "raw", "geih_2025")):
        DATA_DIR = d
        break
else:
    raise FileNotFoundError("No se encontró la carpeta data/raw/geih_2025")

RAW = os.path.join(DATA_DIR, "raw", "geih_2025")
PROCESSED = os.path.join(DATA_DIR, "processed")
os.makedirs(PROCESSED, exist_ok=True)

meses = sorted(os.listdir(RAW))
print("Carpeta de datos:", DATA_DIR)
print("Meses encontrados:", meses)

Carpeta de datos: ../data
Meses encontrados: ['01_enero', '02_febrero', '03_marzo', '04_abril', '05_mayo', '06_junio', '07_julio', '08_agosto', '09_septiembre', '10_octubre', '11_noviembre', '12_diciembre']


## 2. Leer y unir los módulos de cada mes
Solo se leen las columnas necesarias para ahorrar memoria. Los archivos del DANE vienen separados por `;` y en codificación `latin1`.

In [14]:
LLAVE = ["MES", "DIRECTORIO", "SECUENCIA_P", "ORDEN"]

COLUMNAS = {
    "caracteristicas_generales.csv": LLAVE + ["HOGAR", "FEX_C18", "DPTO", "AREA", "CLASE", "P6040", "P3271", "P3042", "P6170"],
    "fuerza_de_trabajo.csv":         LLAVE + ["FT", "FFT"],
    "ocupados.csv":                  LLAVE + ["P6426", "P6430", "P6440", "P6460", "P6920", "INGLABO", "RAMA2D_R4"],
    "no_ocupados.csv":               LLAVE + ["DSI", "P7430", "P7250"],
}

def leer(mes, archivo):
    ruta = os.path.join(RAW, mes, archivo)
    df = pd.read_csv(ruta, sep=";", encoding="latin1", usecols=COLUMNAS[archivo], dtype=str)
    # Algunos meses usan coma decimal: se convierte todo a número
    for c in df.columns:
        df[c] = pd.to_numeric(df[c].str.replace(",", ".", regex=False), errors="coerce")
    return df

partes = []
for mes in meses:
    personas   = leer(mes, "caracteristicas_generales.csv")
    fuerza     = leer(mes, "fuerza_de_trabajo.csv")
    ocupados   = leer(mes, "ocupados.csv")
    no_ocup    = leer(mes, "no_ocupados.csv")

    # Filtrar jóvenes de 18 a 28 años desde el inicio
    jovenes = personas[(personas["P6040"] >= 18) & (personas["P6040"] <= 28)]

    union = (jovenes.merge(fuerza,   on=LLAVE, how="left")
                    .merge(ocupados, on=LLAVE, how="left", indicator="en_ocupados")
                    .merge(no_ocup,  on=LLAVE, how="left"))
    partes.append(union)
    print(f"{mes}: {len(jovenes):,} jóvenes")

geih = pd.concat(partes, ignore_index=True)
print("\nTotal registros de jóvenes 18-28 (12 meses):", f"{len(geih):,}")

01_enero: 11,136 jóvenes
02_febrero: 11,350 jóvenes
03_marzo: 11,200 jóvenes
04_abril: 11,378 jóvenes
05_mayo: 11,568 jóvenes
06_junio: 11,232 jóvenes
07_julio: 11,233 jóvenes
08_agosto: 11,276 jóvenes
09_septiembre: 11,259 jóvenes
10_octubre: 11,263 jóvenes
11_noviembre: 11,319 jóvenes
12_diciembre: 10,949 jóvenes

Total registros de jóvenes 18-28 (12 meses): 135,163


## 3. Condición laboral
- **Ocupado:** la persona aparece en el módulo de Ocupados.
- **Desocupado:** `DSI = 1` en el módulo de No ocupados.
- **Inactivo:** fuera de la fuerza de trabajo (`FFT = 1`) y no desocupado (estudiantes, oficios del hogar, etc.).

Los **inactivos no buscan empleo**, así que no hacen parte de la pregunta del proyecto: solo se conservan ocupados y desocupados.

In [15]:
geih["condicion"] = np.select(
    [geih["en_ocupados"] == "both", geih["DSI"] == 1],
    ["Ocupado", "Desocupado"],
    default="Inactivo")

print(geih["condicion"].value_counts())

geih = geih[geih["condicion"] != "Inactivo"].reset_index(drop=True)
print("Jóvenes en la fuerza de trabajo (ocupados + desocupados):", f"{len(geih):,}")

condicion
Ocupado       70168
Inactivo      49741
Desocupado    15254
Name: count, dtype: int64
Jóvenes en la fuerza de trabajo (ocupados + desocupados): 85,422


## 4. Traducir códigos a categorías
Los códigos siguen el diccionario de datos oficial de la GEIH 2025 del DANE (`data/diccionario/diccionario_datos_geih_2025.xlsx`).

In [16]:
SEXO = {1: "Hombre", 2: "Mujer"}

NIVEL_EDUCATIVO = {1: "Ninguno", 2: "Preescolar", 3: "Básica primaria", 4: "Básica secundaria",
                   5: "Media académica", 6: "Media técnica", 7: "Normalista", 8: "Técnica profesional",
                   9: "Tecnológica", 10: "Universitaria", 11: "Especialización", 12: "Maestría", 13: "Doctorado"}

NIVEL_GRUPO = {1: "Básica o menos", 2: "Básica o menos", 3: "Básica o menos", 4: "Básica o menos",
               5: "Bachiller", 6: "Bachiller", 7: "Bachiller", 8: "Técnico", 9: "Tecnólogo",
               10: "Universitario", 11: "Posgrado", 12: "Posgrado", 13: "Posgrado"}

CIUDAD = {5: "Medellín A.M.", 8: "Barranquilla A.M.", 11: "Bogotá", 13: "Cartagena", 15: "Tunja",
          17: "Manizales A.M.", 18: "Florencia", 19: "Popayán", 20: "Valledupar", 23: "Montería",
          27: "Quibdó", 41: "Neiva", 44: "Riohacha", 47: "Santa Marta", 50: "Villavicencio",
          52: "Pasto", 54: "Cúcuta A.M.", 63: "Armenia", 66: "Pereira A.M.", 68: "Bucaramanga A.M.",
          70: "Sincelejo", 73: "Ibagué", 76: "Cali A.M.", 81: "Arauca", 85: "Yopal", 86: "Mocoa",
          88: "San Andrés", 91: "Leticia", 94: "Inírida", 95: "San José del Guaviare", 97: "Mitú",
          99: "Puerto Carreño"}

DEPARTAMENTO = {5: "Antioquia", 8: "Atlántico", 11: "Bogotá D.C.", 13: "Bolívar", 15: "Boyacá",
                17: "Caldas", 18: "Caquetá", 19: "Cauca", 20: "Cesar", 23: "Córdoba", 25: "Cundinamarca",
                27: "Chocó", 41: "Huila", 44: "La Guajira", 47: "Magdalena", 50: "Meta", 52: "Nariño",
                54: "Norte de Santander", 63: "Quindío", 66: "Risaralda", 68: "Santander", 70: "Sucre",
                73: "Tolima", 76: "Valle del Cauca", 81: "Arauca", 85: "Casanare", 86: "Putumayo",
                88: "San Andrés", 91: "Amazonas", 94: "Guainía", 95: "Guaviare", 97: "Vaupés", 99: "Vichada"}

POSICION = {1: "Empleado empresa particular", 2: "Empleado del gobierno", 3: "Empleado doméstico",
            4: "Cuenta propia", 5: "Patrón o empleador", 6: "Trabajador familiar sin remuneración",
            7: "Jornalero o peón", 8: "Otro"}

def sector_ciiu(cod):
    """Agrupa la rama CIIU Rev. 4 (2 dígitos) en sectores económicos."""
    if pd.isna(cod): return np.nan
    c = int(cod)
    tabla = [(3, "Agropecuario"), (9, "Minería"), (33, "Industria manufacturera"), (39, "Servicios públicos"),
             (43, "Construcción"), (47, "Comercio"), (53, "Transporte y logística"),
             (56, "Alojamiento y comida"), (63, "Información y comunicaciones (TIC)"),
             (66, "Servicios financieros"), (68, "Inmobiliarias"), (75, "Actividades profesionales"),
             (82, "Servicios administrativos y de apoyo"), (84, "Administración pública"),
             (85, "Educación"), (88, "Salud"), (93, "Arte y entretenimiento"), (96, "Otros servicios"),
             (98, "Hogares como empleadores"), (99, "Organismos extraterritoriales")]
    for limite, nombre in tabla:
        if c <= limite:
            return nombre
    return np.nan

## 5. Construir la tabla limpia

In [17]:
df = pd.DataFrame({
    "mes": geih["MES"].astype(int),
    "id_hogar": geih["DIRECTORIO"].astype(int).astype(str) + "-" + geih["SECUENCIA_P"].astype(int).astype(str),
    "orden": geih["ORDEN"].astype(int),
    # Factor de expansión: al unir 12 meses se divide entre 12 para obtener promedios anuales
    "factor_expansion": (geih["FEX_C18"] / 12).round(2),
    "codigo_departamento": geih["DPTO"].astype(int),   # código DIVIPOLA: llave para cruzar con el SPE
    "departamento": geih["DPTO"].map(DEPARTAMENTO),
    "ciudad": geih["AREA"].map(CIUDAD).fillna("Resto del país"),
    "zona": geih["CLASE"].map({1: "Urbana (cabecera)", 2: "Rural y centros poblados"}),
    "edad": geih["P6040"].astype(int),
    "sexo": geih["P3271"].map(SEXO),
    "nivel_educativo": geih["P3042"].map(NIVEL_EDUCATIVO),
    "nivel_educativo_grupo": geih["P3042"].map(NIVEL_GRUPO),
    "estudia_actualmente": geih["P6170"].map({1: "Sí", 2: "No"}),
    "condicion": geih["condicion"],
    # --- Solo desocupados ---
    "ha_trabajado_antes": geih["P7430"].map({1: "Sí", 2: "No"}).where(geih["condicion"] == "Desocupado"),
    "semanas_buscando": geih["P7250"].where((geih["condicion"] == "Desocupado") & (geih["P7250"] < 998)).astype("Int64"),  # 998/999 = no sabe
    # --- Solo ocupados ---
    "meses_en_empleo_actual": geih["P6426"].where(geih["condicion"] == "Ocupado").astype("Int64"),
    "posicion_ocupacional": geih["P6430"].map(POSICION),
    "tiene_contrato": geih["P6440"].map({1: "Sí", 2: "No"}),
    "tipo_contrato": geih["P6460"].map({1: "Término indefinido", 2: "Término fijo"}),
    "cotiza_pension": geih["P6920"].map({1: "Sí", 2: "No", 3: "Ya es pensionado"}),
    "ingreso_laboral_mensual": geih["INGLABO"].round().astype("Int64"),
    "sector": geih["RAMA2D_R4"].apply(sector_ciiu),
})

# Empleo formal (aproximación): cotiza a pensión
df["empleo_formal"] = (df["cotiza_pension"].isin(["Sí", "Ya es pensionado"])
                        .map({True: "Sí", False: "No"})
                        .where(df["condicion"] == "Ocupado"))
df.head()

,mes,id_hogar,orden,factor_expansion,codigo_departamento,departamento,ciudad,zona,edad,sexo,nivel_educativo,nivel_educativo_grupo,estudia_actualmente,condicion,ha_trabajado_antes,semanas_buscando,meses_en_empleo_actual,posicion_ocupacional,tiene_contrato,tipo_contrato,cotiza_pension,ingreso_laboral_mensual,sector,empleo_formal
0,1,8086887-1,1,26.93,13,Bolívar,Resto del país,Rural y centros poblados,26,Mujer,Básica secundaria,Básica o menos,No,Ocupado,NaN,<NA>,60,Cuenta propia,No,NaN,No,600000,Servicios administrativos y de apoyo,No
1,1,8086893-1,5,260.42,13,Bolívar,Resto del país,Urbana (cabecera),28,Hombre,Universitaria,Universitario,No,Ocupado,NaN,<NA>,11,Cuenta propia,Sí,Término fijo,Sí,1123000,Educación,Sí
2,1,8086897-1,1,232.31,13,Bolívar,Resto del país,Urbana (cabecera),20,Mujer,Media académica,Bachiller,No,Desocupado,Sí,4,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
3,1,8086923-1,3,14.44,18,Caquetá,Resto del país,Rural y centros poblados,25,Mujer,Universitaria,Universitario,Sí,Ocupado,NaN,<NA>,48,Trabajador familiar sin remuneración,No,NaN,No,<NA>,Arte y entretenimiento,No
4,1,8086930-1,1,54.25,18,Caquetá,Resto del país,Rural y centros poblados,22,Hombre,Básica primaria,Básica o menos,No,Ocupado,NaN,<NA>,1,Empleado empresa particular,Sí,NaN,No,1400000,Agropecuario,No


## 6. Variable objetivo: jóvenes sin experiencia que consiguieron empleo
La GEIH no pregunta directamente “¿es su primer empleo?”, así que usamos una **aproximación** (documentada como riesgo en la ficha):

| Grupo | Definición | `consiguio_empleo` |
|---|---|---|
| **Aspirantes** | Desocupados que **nunca han trabajado** (`P7430 = 2`) | 0 |
| **Recién insertados** | Ocupados con **12 meses o menos** en su empleo actual (`P6426 ≤ 12`) | 1 |

Los demás jóvenes (ocupados con más de 12 meses y desocupados que ya trabajaron) quedan en la tabla, pero con `sin_experiencia = 0` y la variable objetivo vacía.

In [18]:
aspirante = (df["condicion"] == "Desocupado") & (df["ha_trabajado_antes"] == "No")
recien    = (df["condicion"] == "Ocupado") & (df["meses_en_empleo_actual"] <= 12)

df["sin_experiencia"] = (aspirante | recien).astype(int)
df["consiguio_empleo"] = np.select([recien, aspirante], [1, 0], default=np.nan)

print(pd.crosstab(df["condicion"], df["consiguio_empleo"].fillna(-1).map({1: "1 (recién insertado)", 0: "0 (aspirante)", -1: "fuera del grupo"})))

consiguio_empleo  0 (aspirante)  1 (recién insertado)  fuera del grupo
condicion                                                             
Desocupado                 4344                     0            10910
Ocupado                       0                 37441            32727


## 7. Revisión de calidad

In [19]:
# Duplicados en la llave de persona
dup = df.duplicated(subset=["mes", "id_hogar", "orden"]).sum()
print("Registros duplicados:", dup)

# Datos faltantes (las columnas de ocupados/desocupados están vacías a propósito para los demás grupos)
faltantes = df.isna().mean().mul(100).round(1).to_frame("% faltante")
faltantes[faltantes["% faltante"] > 0]

Registros duplicados: 0


,% faltante
ha_trabajado_antes,82.1
semanas_buscando,82.2
meses_en_empleo_actual,17.9
posicion_ocupacional,17.9
tiene_contrato,17.9
tipo_contrato,65.0
cotiza_pension,17.9
ingreso_laboral_mensual,19.7
sector,17.9
empleo_formal,17.9


In [20]:
# Ingresos: 0 o vacío es normal (trabajadores sin remuneración o sin respuesta)
df.loc[df["condicion"] == "Ocupado", "ingreso_laboral_mensual"].describe().round(0)

,ingreso_laboral_mensual
count,68632.0
mean,1452044.0
std,1250097.0
min,0.0
25%,800000.0
50%,1423500.0
75%,1623500.0
max,103000000.0


## 8. Primer vistazo con factores de expansión
Los porcentajes se calculan **ponderados** con `factor_expansion` para representar a la población, no solo a la muestra.

In [21]:
base = df[df["sin_experiencia"] == 1]

def tasa_ponderada(col):
    t = (base.groupby(col)
             .apply(lambda g: pd.Series({
                 "registros": len(g),
                 "jovenes_estimados": g["factor_expansion"].sum(),
                 "tasa_empleo_%": np.average(g["consiguio_empleo"], weights=g["factor_expansion"]) * 100}),
                 include_groups=False)
             .sort_values("tasa_empleo_%", ascending=False))
    return t.round(1)

print(f"Jóvenes sin experiencia en la muestra: {len(base):,}")
print(f"Jóvenes sin experiencia estimados (promedio mensual): {base['factor_expansion'].sum():,.0f}")
tasa_ponderada("nivel_educativo_grupo")

Jóvenes sin experiencia en la muestra: 41,785
Jóvenes sin experiencia estimados (promedio mensual): 2,846,838


,registros,jovenes_estimados,tasa_empleo_%
nivel_educativo_grupo,,,
Básica o menos,5781.0,440522.8,96.4
Posgrado,697.0,44068.5,95.5
Bachiller,18363.0,1320383.1,92.1
Tecnólogo,2291.0,151327.3,90.2
Universitario,9596.0,583981.9,89.4
Técnico,5056.0,306532.5,89.3


In [22]:
tasa_ponderada("sexo")

,registros,jovenes_estimados,tasa_empleo_%
sexo,,,
Hombre,21626.0,1575008.6,93.2
Mujer,20159.0,1271829.1,90.1


In [23]:
tasa_ponderada("ciudad")

,registros,jovenes_estimados,tasa_empleo_%
ciudad,,,
Mitú,388.0,919.7,97.4
Villavicencio,1585.0,36273.8,96.1
Armenia,1279.0,17245.4,95.7
Yopal,360.0,10618.8,95.6
San José del Guaviare,432.0,4314.5,95.2
Medellín A.M.,2062.0,263540.2,94.9
Pereira A.M.,1708.0,40056.5,94.7
Manizales A.M.,1747.0,29181.0,94.4
Bogotá,1823.0,549384.0,93.8


## 9. Guardar el resultado

In [24]:
salida = os.path.join(PROCESSED, "geih2025_jovenes_18_28.csv")
df.to_csv(salida, index=False, encoding="utf-8-sig")
print("Guardado en:", salida)
print("Filas:", len(df), "| Columnas:", df.shape[1])

Guardado en: ../data/processed/geih2025_jovenes_18_28.csv
Filas: 85422 | Columnas: 26
